In [4]:
import os, torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from models import FaceCNN, face_eval_tf, FACE_LABELS

os.makedirs("saved", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = "data"

train_tf = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((48, 48)),
    transforms.RandomResizedCrop(48, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.RandomAutocontrast(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
train_ds = datasets.ImageFolder(f"{ROOT}/train", transform=train_tf)
test_ds = datasets.ImageFolder(f"{ROOT}/test", transform=face_eval_tf)
print(train_ds.classes)
assert train_ds.classes == FACE_LABELS

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=256, num_workers=0)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [5]:
model = FaceCNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 25
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, steps_per_epoch=len(train_dl), epochs=EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

def evaluate():
    model.eval(); c = t = 0; preds = []; ys = []
    with torch.no_grad():
        for x, y in test_dl:
            p = model(x.to(device)).argmax(1).cpu()
            c += (p == y).sum().item(); t += len(y)
            preds += p.tolist(); ys += y.tolist()
    return c / t, ys, preds

best = 0
for epoch in range(EPOCHS):
    model.train(); tot = 0
    for x, y in train_dl:
        opt.zero_grad()
        loss = loss_fn(model(x.to(device)), y.to(device))
        loss.backward(); opt.step(); sched.step()
        tot += loss.item()
    acc, _, _ = evaluate()
    print(f"epoch {epoch+1} loss {tot/len(train_dl):.4f} test_acc {acc:.4f}")
    if acc > best:
        best = acc
        torch.save(model.state_dict(), "saved/face_cnn.pth")

epoch 1 loss 1.8296 test_acc 0.2910
epoch 2 loss 1.7308 test_acc 0.4042
epoch 3 loss 1.6571 test_acc 0.4604
epoch 4 loss 1.6141 test_acc 0.4805
epoch 5 loss 1.5842 test_acc 0.4778
epoch 6 loss 1.5536 test_acc 0.4792
epoch 7 loss 1.5295 test_acc 0.5047
epoch 8 loss 1.5014 test_acc 0.5266
epoch 9 loss 1.4787 test_acc 0.5344
epoch 10 loss 1.4574 test_acc 0.5581
epoch 11 loss 1.4428 test_acc 0.5305
epoch 12 loss 1.4335 test_acc 0.5816
epoch 13 loss 1.4090 test_acc 0.5777
epoch 14 loss 1.3977 test_acc 0.5783
epoch 15 loss 1.3872 test_acc 0.5793
epoch 16 loss 1.3663 test_acc 0.5880
epoch 17 loss 1.3562 test_acc 0.6030
epoch 18 loss 1.3411 test_acc 0.6017
epoch 19 loss 1.3319 test_acc 0.6002
epoch 20 loss 1.3193 test_acc 0.6071
epoch 21 loss 1.3080 test_acc 0.6085
epoch 22 loss 1.2976 test_acc 0.6204
epoch 23 loss 1.2982 test_acc 0.6184
epoch 24 loss 1.2954 test_acc 0.6187
epoch 25 loss 1.2878 test_acc 0.6186


In [6]:
model.load_state_dict(torch.load("saved/face_cnn.pth", map_location=device))
acc, ys, preds = evaluate()
print("best test acc:", acc)
print(classification_report(ys, preds, target_names=FACE_LABELS, zero_division=0))

best test acc: 0.6203677904708833
              precision    recall  f1-score   support

       angry       0.53      0.55      0.54       958
     disgust       1.00      0.01      0.02       111
        fear       0.50      0.25      0.34      1024
       happy       0.82      0.87      0.84      1774
     neutral       0.52      0.72      0.60      1233
         sad       0.50      0.49      0.49      1247
    surprise       0.75      0.77      0.76       831

    accuracy                           0.62      7178
   macro avg       0.66      0.52      0.51      7178
weighted avg       0.62      0.62      0.60      7178

